

## Load NAB time series

## Step 1 - Load ambient dataset and attach anomaly labels

In this step, the raw `ambient_temperature_system_failure.csv` file from the NAB
`realKnownCause` folder is loaded into a dataframe. The corresponding anomaly
timestamps are then retrieved from `combined_labels.json` and converted into a
binary `is_anomaly` column. The timestamp column is parsed as a proper datetime
type and the rows are sorted chronologically so that each record reflects the
correct order in time.


### A) Load raw ambient_temperature_system_failure.csv dataset

In [9]:
## Imports
import pandas as pd
import numpy as np

In [10]:
import os

# Path to Numeta Anomoly Benchmark (NAB) data folder
base_path = "../data/NAB-master/data/realKnownCause/"

# Load one example file
file_path = os.path.join(base_path, "ambient_temperature_system_failure.csv")
df = pd.read_csv(file_path)

df.head()


,timestamp,value
0,2013-07-04 00:00:00,69.880835
1,2013-07-04 01:00:00,71.220227
2,2013-07-04 02:00:00,70.877805
3,2013-07-04 03:00:00,68.959400
4,2013-07-04 04:00:00,69.283551


### B) Load Anomaly Labels (JSON)

In [11]:
import json  

# Where the labels file is
label_path = "../data/NAB-master/labels/combined_labels.json"

# Open the file and read its contents
with open(label_path, "r") as label_file:      # "r" = read mode, label_file = name for the opened file
    labels = json.load(label_file)             # turn the JSON into a Python dictionary called 'labels'

# Shows the first 10 dataset keys names to confirm it loaded correctly
list(labels.keys())[:10]


['artificialNoAnomaly/art_daily_no_noise.csv',
 'artificialNoAnomaly/art_daily_perfect_square_wave.csv',
 'artificialNoAnomaly/art_daily_small_noise.csv',
 'artificialNoAnomaly/art_flatline.csv',
 'artificialNoAnomaly/art_noisy.csv',
 'artificialWithAnomaly/art_daily_flatmiddle.csv',
 'artificialWithAnomaly/art_daily_jumpsdown.csv',
 'artificialWithAnomaly/art_daily_jumpsup.csv',
 'artificialWithAnomaly/art_daily_nojump.csv',
 'artificialWithAnomaly/art_increase_spike_density.csv']

### C) Extract Labels for Selected Dataset

In [12]:
# Select the correct dataset key from the labels dictionary
dataset_key = "realKnownCause/ambient_temperature_system_failure.csv"

# Extract the list of anomaly timestamps for this dataset
ambient_label_times = labels[dataset_key]

# Turn the list of timestamp strings into a DataFrame
ambient_label_times_df = pd.DataFrame(
    {"timestamp": pd.to_datetime(ambient_label_times)}
)

# Display the timestamps
ambient_label_times_df.head()

,timestamp
0,2013-12-22 20:00:00
1,2014-04-13 09:00:00


### D) Add Labels to DataFrame

In [13]:
# Converting the label timestamps into datetime objects 
ambient_label_times = pd.to_datetime(labels[dataset_key])

# D) Ensure timestamp is datetime and sorted
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Create a new column: 1 if the timestamp is an anomaly, 0 otherwise
df["is_anomaly"] = df["timestamp"].isin(ambient_label_times).astype(int)

# See the updated dataframe
df.head()


,timestamp,value,is_anomaly
0,2013-07-04 00:00:00,69.880835,0
1,2013-07-04 01:00:00,71.220227,0
2,2013-07-04 02:00:00,70.877805,0
3,2013-07-04 03:00:00,68.959400,0
4,2013-07-04 04:00:00,69.283551,0


### E) Class Balance

In [14]:
# Class balance for the is_anomaly column
anomaly_balance = (
    df["is_anomaly"]
      .value_counts()                            # counts for 0 and 1
      .rename(index={0: "Normal", 1: "Anomaly"}) # rename classes for readability
      .to_frame(name="Count")                    # make it a DataFrame with a 'Count' column
      .reset_index()                             # turn index into a normal column
      .rename(columns={"is_anomaly": "Class"})       
)

# Percentage column
anomaly_balance["Proportion (%)"] = (
    anomaly_balance["Count"] / len(df) * 100
).round(3)

anomaly_balance


,Class,Count,Proportion (%)
0,Normal,7265,99.972
1,Anomaly,2,0.028


## Step 2 - Standardise core columns (`time`, `value`, `is_anomaly`, `case_study`)

In [15]:
# Ensure timestamp is datetime and sorted
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Rename to common research project schema
df = df.rename(columns={"timestamp": "time"})

# Add a case_study identifier
df["case_study"] = "ambient"

# Quick check
df[["time", "value", "is_anomaly", "case_study"]].head()


,time,value,is_anomaly,case_study
0,2013-07-04 00:00:00,69.880835,0,ambient
1,2013-07-04 01:00:00,71.220227,0,ambient
2,2013-07-04 02:00:00,70.877805,0,ambient
3,2013-07-04 03:00:00,68.959400,0,ambient
4,2013-07-04 04:00:00,69.283551,0,ambient


###  Quick verification

In [19]:
df[["time","value","is_anomaly","case_study"]].dtypes

time          datetime64[ns]
value                float64
is_anomaly             int64
case_study            object
dtype: object

In this step, the ambient temperature dataframe is aligned with the common schema used across all
case studies. The `timestamp` column is converted to a proper datetime type, the rows are sorted
chronologically, and the column is renamed to `time`. A `case_study` column with the value
`"ambient"` is added so that this dataset can later be stacked or filtered alongside the other case
studies using the same core columns (`time`, `value`, `is_anomaly`, `case_study`).


## Step 3 - Missing values and Duplicates rows sanity check

### A) Missing values per column


In [16]:
# Count missing values in each column
missing_counts = df.isna().sum()

# Percentage of missing values in each column
missing_percentages = (df.isna().mean() * 100).round(3)

# Combine into a summary table
missing_summary = (
    pd.DataFrame(
        {
            "Column": missing_counts.index,
            "Missing count": missing_counts.values,
            "Missing (%)": missing_percentages.values,
        }
    )
    .sort_values("Missing count", ascending=False)
    .reset_index(drop=True)
)

missing_summary

,Column,Missing count,Missing (%)
0,time,0,0.0
1,value,0,0.0
2,is_anomaly,0,0.0
3,case_study,0,0.0


### B) Duplicate rows summary 

In [21]:
# Duplicate rows summary
duplicate_count = df.duplicated().sum()
total_rows = len(df)

duplicate_summary = pd.DataFrame(
    [
        ("Total rows", total_rows),
        ("Number of duplicate rows", duplicate_count),
        (
            "Duplicate rows (%)",
            round(duplicate_count / total_rows * 100, 3),
        ),
    ],
    columns=["Statistic", "Value"],
)

duplicate_summary


,Statistic,Value
0,Total rows,7267.0
1,Number of duplicate rows,0.0
2,Duplicate rows (%),0.0


The missing-values table confirms that the ambient temperature dataset has no missing entries in
any of the core columns (`time`, `value`, `is_anomaly`, `case_study`). The duplicate-rows summary
also shows zero duplicate records. These checks indicate that the raw NAB file for ambient
temperature is already clean, and no imputation or de-duplication is required before modelling.
Recording this explicitly makes the preprocessing pipeline transparent for later write-up.


## Step 4 - Convert `value` from Fahrenheit to Celsius 


In [22]:
# Convert the main `value` column from Fahrenheit to Celsius in place
df["value"] = (df["value"] - 32.0) * 5.0 / 9.0

# Quick sanity check
df[["time", "value", "is_anomaly", "case_study"]].head()
#df["value"].describe()


,time,value,is_anomaly,case_study
0,2013-07-04 00:00:00,21.044908,0,ambient
1,2013-07-04 01:00:00,21.789015,0,ambient
2,2013-07-04 02:00:00,21.598781,0,ambient
3,2013-07-04 03:00:00,20.533000,0,ambient
4,2013-07-04 04:00:00,20.713084,0,ambient


### Temperature conversion from Fahrenheit to Celsius

The ambient temperature values in the original NAB file are recorded in
Fahrenheit. For this project, all temperatures were converted to Celsius and the
`value` column in the processed dataset is expressed in degrees Celsius (°C).

This choice was made for three main reasons:

- **Contextual alignment**: The research is situated in a South African context,
  where Celsius is the standard unit used in practice (e.g. in weather reports,
  building management systems, and operational monitoring). Expressing the
  series in °C makes the results easier to interpret for local readers and
  examiners.

- **Interpretability in reporting**: Descriptive statistics, plots, and anomaly
  windows can be discussed directly in Celsius without repeatedly converting or
  annotating Fahrenheit values. This simplifies the narrative in the data
  overview and results chapters while keeping the underlying time series
  unchanged apart from a linear unit transformation.

- **Consistency across the preprocessing pipeline**: By converting once, early
  in the preprocessing, and overwriting the main `value` column, all subsequent
  steps (scaling, feature creation, model inputs, and evaluation) work with a
  single, clearly defined temperature scale. The conversion is invertible and
  does not alter the temporal structure, anomaly locations, or overall shape of
  the series; it only changes the unit of measurement to better match the
  research context.


### Temperature Summary Table 

In [23]:
# Raw summary stats for Celsius values
value_summary = df["value"].describe()

# Turn into a one-column DataFrame and clean up labels
value_summary = (
    value_summary
    .to_frame(name="Ambient Temperature (°C)")
    .rename(index={
        "count": "Count",
        "mean": "Mean",
        "std":  "Std. deviation",
        "min":  "Min",
        "25%":  "Q1 (25%)",
        "50%":  "Median (50%) ",
        "75%":  "Q3 (75%)",
        "max":  "Max",
    })
    .round(2)
)

value_summary

,Ambient Temperature (°C)
Count,7267.00
Mean,21.80
Std. deviation,2.36
Min,14.14
Q1 (25%),20.21
Median (50%),22.14
Q3 (75%),23.57
Max,30.12


## Step 5 - Create basic time-based features

In [24]:
# Hour of day (0–23)
df["hour_of_day"] = df["time"].dt.hour

# Day of week (0 = Monday, 6 = Sunday)
df["day_of_week"] = df["time"].dt.dayofweek

# Weekend flag (Saturday = 5, Sunday = 6)
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# Quick check
df[["time", "hour_of_day", "day_of_week", "is_weekend"]].head()


,time,hour_of_day,day_of_week,is_weekend
0,2013-07-04 00:00:00,0,3,0
1,2013-07-04 01:00:00,1,3,0
2,2013-07-04 02:00:00,2,3,0
3,2013-07-04 03:00:00,3,3,0
4,2013-07-04 04:00:00,4,3,0


This step derives simple calendar features from the `time` column to provide
additional structure for later analysis and modelling. The timestamp is broken
down into:

- `hour_of_day`: the hour of the day (0–23),
- `day_of_week`: the day index (0 = Monday, …, 6 = Sunday),
- `is_weekend`: an indicator variable that is 1 for Saturday and Sunday, and 0
  for weekdays.

These features do not change the underlying temperature values, but they allow
later models and visualisations to capture patterns that depend on time of day
or weekday/weekend effects.


## Step 6 - Reorder columns for a clean schema

In [25]:
# Desired column order
column_order = [
    "time",
    "value",
    "is_anomaly",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "case_study",
]

# Reorder the dataframe
df = df[column_order]

df.head()


,time,value,is_anomaly,hour_of_day,day_of_week,is_weekend,case_study
0,2013-07-04 00:00:00,21.044908,0,0,3,0,ambient
1,2013-07-04 01:00:00,21.789015,0,1,3,0,ambient
2,2013-07-04 02:00:00,21.598781,0,2,3,0,ambient
3,2013-07-04 03:00:00,20.533000,0,3,3,0,ambient
4,2013-07-04 04:00:00,20.713084,0,4,3,0,ambient


## Step 7 - Inspect time span and anomaly distribution (before data split)

In [26]:
# Overall time span
start_time = df["time"].min()
end_time = df["time"].max()
n_rows = len(df)
duration = end_time - start_time

time_span_summary = pd.DataFrame(
    [
        ("Start time", start_time),
        ("End time", end_time),
        ("Number of rows", n_rows),
        ("Total duration (days)", duration.days),
    ],
    columns=["Statistic", "Value"],
)

time_span_summary


,Statistic,Value
0,Start time,2013-07-04 00:00:00
1,End time,2014-05-28 15:00:00
2,Number of rows,7267
3,Total duration (days),328


In [27]:
# Summary table for the ambient anomalies
anomaly_summary = (
    df.loc[df["is_anomaly"] == 1, ["time", "value"]]
      .reset_index()  # keep original row index for reference
      .rename(columns={
          "index": "Row index",
          "time": "Anomaly time",
          "value": "Temperature (°C)",
      })
      .round({"Temperature (°C)": 2})
)

anomaly_summary


,Row index,Anomaly time,Temperature (°C)
0,3721,2013-12-22 20:00:00,30.11
1,6180,2014-04-13 09:00:00,14.14


These summaries provide a clear view of how long the series runs and how many
observations it contains, as well as where the rare anomaly events occur in
time. This context will guide the choice of chronological split points in the
next step.


### Quick internal check of sampling interval




In [28]:
# Compute differences between consecutive timestamps
time_difference = df["time"].diff().dropna()

# Identify the most common time difference
most_common_difference = time_difference.value_counts().idxmax()

most_common_difference


Timedelta('0 days 01:00:00')

A detailed analysis of the sampling interval and any gaps in the series is
already provided in the Data Overview notebook for the ambient dataset.
Here, we only perform a minimal internal check to confirm that the dominant
interval between observations is **one hour**, consistent with the agreed
assumptions for this case study.

## Step 8 - Define a first draft of train /validation /test splits

### A) Define training and validation boundaries using anomaly times


In [30]:
# Extract anomaly times from the anomaly_summary table
first_anomaly_time = anomaly_summary.loc[0, "Anomaly time"]
second_anomaly_time = anomaly_summary.loc[1, "Anomaly time"]

# Place split boundaries 7 days before each anomaly
training_end_time = first_anomaly_time - pd.Timedelta(days=7)
validation_end_time = second_anomaly_time - pd.Timedelta(days=7)

# Summary table of the chosen boundaries and anomaly times
split_boundaries = pd.DataFrame(
    [
        ("Training end time", training_end_time),
        ("Validation end time", validation_end_time),
        ("First anomaly time", first_anomaly_time),
        ("Second anomaly time", second_anomaly_time),
    ],
    columns=["Description", "Time"],
)

split_boundaries


,Description,Time
0,Training end time,2013-12-15 20:00:00
1,Validation end time,2014-04-06 09:00:00
2,First anomaly time,2013-12-22 20:00:00
3,Second anomaly time,2014-04-13 09:00:00


In this step, the timestamps of the two labelled anomalies are used to define
chronological boundaries for the training and validation segments.

The anomaly times are
stored as `first_anomaly_time` and `second_anomaly_time`. 
The end of the
training period is then set to seven days before the first anomaly, and the end
of the validation period is set to seven days before the second anomaly. This
one-week buffer is intended to keep the training data focused on clearly normal
behaviour, while reserving the neighbourhood of each anomaly for validation and
test evaluation.

The table reports the resulting training and validation end
times alongside the original anomaly times. This provides a transparent record
of how the temporal split points were chosen and can be cited directly in the
methodology or data preparation section of the thesis.


### B) Assign split labels and summarise train/validation/test segments


In [31]:
# Helper function to assign each timestamp to a split
def assign_split(time_value):
    if time_value <= training_end_time:
        return "train"
    elif time_value <= validation_end_time:
        return "validation"
    else:
        return "test"

# Add the split label column to the dataframe
df["split"] = df["time"].apply(assign_split)

# Total number of rows for proportion calculation
total_rows = len(df)

# Summarise rows and anomalies per split (train, validation and test)
split_summary = (
    df.groupby("split")
      .agg(
          Start_time=("time", "min"),
          End_time=("time", "max"),
          Rows=("time", "size"),
          Anomalies=("is_anomaly", "sum"),
      )
      .reset_index()
)

# Desired ordering: train, validation, test
split_order = ["train", "validation", "test"]
split_summary["split"] = pd.Categorical(
    split_summary["split"],
    categories=split_order,
    ordered=True,
)
split_summary = split_summary.sort_values("split") 

# Add a proportion column (percentage of the full dataset)
split_summary["Proportion (%)"] = (
    split_summary["Rows"] / total_rows * 100
).round(2)

split_summary


,split,Start_time,End_time,Rows,Anomalies,Proportion (%)
1,train,2013-07-04 00:00:00,2013-12-15 20:00:00,3554,0,48.91
2,validation,2013-12-15 21:00:00,2014-04-03 09:00:00,2560,1,35.23
0,test,2014-04-10 15:00:00,2014-05-28 15:00:00,1153,1,15.87



- In step 8 the previously defined variables `training_end_time` and `validation_end_time` are used to
  classify each timestamp into one of three chronological segments:
  - `train` – earliest period, ending before the first anomaly buffer.
  - `validation` – middle period, ending before the second anomaly buffer.
  - `test` – most recent period, after the validation boundary.

- Apply a small helper function to the `time` column to:
  - assign a split label (`"train"`, `"validation"`, `"test"`) to every row, and
  - store the result in a new `split` column in the dataframe.

- Group the data by `split` to create a compact summary table reporting, for
  each segment:
  - start and end timestamps,
  - total number of rows,
  - number of anomaly points (`is_anomaly = 1`),
  - proportion of the full dataset represented by that segment.

- Enforce the logical ordering `train → validation → test` in the summary table
  so that results are easy to read and can be directly cited in the methodology
  and results chapters.


### C) Create split-specific dataframes and verify no rows are lost

The `split` column now tags each row in the ambient dataframe as belonging to
the training, validation, or test segment. In this step, the full processed
ambient dataframe is kept intact, and separate dataframes for each split are
created using boolean masks on the `split` column.

A small integrity check is then used to confirm that the number of rows in
`train_df`, `validation_df`, and `test_df` adds up exactly to the number of
rows in the full dataframe. This ensures that the chronological splitting does
not accidentally discard any data and can be safely re-run without shrinking
the dataset.


In [32]:
# Keep track of the total number of rows in the full processed ambient dataframe
total_rows_full = len(df)

# Create non-destructive split-specific dataframes using boolean masks
train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

In [33]:
# Row counts in each split dataframe
n_train = len(train_df)
n_validation = len(validation_df)
n_test = len(test_df)

# Build an examiner-style integrity summary to check that no rows are lost
split_integrity_summary = pd.DataFrame(
    [
        ("Total rows in full df", total_rows_full),
        ("Rows in train_df", n_train),
        ("Rows in validation_df", n_validation),
        ("Rows in test_df", n_test),
        ("Sum of split rows", n_train + n_validation + n_test),
    ],
    columns=["Statistic", "Value"],
)

split_integrity_summary


,Statistic,Value
0,Total rows in full df,7267
1,Rows in train_df,3554
2,Rows in validation_df,2560
3,Rows in test_df,1153
4,Sum of split rows,7267


## Reorder columns for a clean schema

In [34]:
# Desired column order
column_order = [
    "time",
    "value",
    "is_anomaly",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "split",
    "case_study",
]

# Reorder the dataframe
df = df[column_order]

df.head()


,time,value,is_anomaly,hour_of_day,day_of_week,is_weekend,split,case_study
0,2013-07-04 00:00:00,21.044908,0,0,3,0,train,ambient
1,2013-07-04 01:00:00,21.789015,0,1,3,0,train,ambient
2,2013-07-04 02:00:00,21.598781,0,2,3,0,train,ambient
3,2013-07-04 03:00:00,20.533000,0,3,3,0,train,ambient
4,2013-07-04 04:00:00,20.713084,0,4,3,0,train,ambient


## Step 9 - Standardise temperature values using only the training split


### A) Fit scaler on training and store parameters

In [35]:
from sklearn.preprocessing import StandardScaler

# Fit a StandardScaler using ONLY the training split (Celsius values)
scaler = StandardScaler()
scaler.fit(train_df[["value"]])  # train_df is df[df["split"] == "train"]

# Store the training mean and std used for scaling (for documentation/reproducibility)
ambient_scaler_parameters = {
    "mean": float(scaler.mean_[0]),   # mean temperature (°C) on training split
    "std": float(scaler.scale_[0]),   # std dev (°C) on training split
}

ambient_scaler_parameters # StandardScaler result from learning training data

{'mean': 22.362580421515663, 'std': 1.8408455345180461}

### B) Apply scaling to full dataframe and check training behaviour

In [36]:
# Apply the fitted scaler to the FULL ambient dataframe (train, validation, test)
df["value_scaled"] = scaler.transform(df[["value"]])

# Describe the scaled values on the training split to confirm mean ~ 0 and std ~ 1
scaled_training_summary = (
    df[df["split"] == "train"]["value_scaled"]
      .describe()[["mean", "std"]]
      .to_frame(name="Training value_scaled")
      .rename(index={"mean": "Mean", "std": "Std. deviation"})
      .round(4)
)

scaled_training_summary

,Training value_scaled
Mean,0.0000
Std. deviation,1.0001


### C) Summary of value_scaled by split

In [37]:
# Summary of scaled values by split (train, validation, test)
split_scaled_summary = (
    df.groupby("split")["value_scaled"]
      .describe()[["mean", "std", "min", "max"]]
      .round(3)
      .reset_index()
)

split_scaled_summary


,split,mean,std,min,max
0,test,-1.830,1.042,-4.465,0.752
1,train,0.000,1.000,-3.286,2.108
2,validation,-0.041,1.242,-3.392,4.216


##  Standardising the ambient temperature values

- **Cell A - Fit the scaler on the training split and store parameters**
  - Fits `StandardScaler` using only the `value` column from `train_df`.
  - Learns the **average temperature** and **spread (standard deviation)** of the training data in °C.
  - Saves these numbers in `ambient_scaler_parameters = {"mean": ..., "std": ...}` so the scaling step is
    transparent and can be reported or reproduced later.
  - This respects the rule that all preprocessing statistics should be based on the **training split only**, to
    avoid leaking information from validation/test into training.

- **Cell B - Apply scaling to the full dataframe and verify behaviour on training**
  - Uses the fitted scaler to create a new column `value_scaled` for **all rows** in `df`
    (train, validation, and test).
  - Checks that, on the **training split**, `value_scaled` has mean ≈ 0 and standard deviation ≈ 1.
  - This confirms that the models will see a temperature feature that is on a standard scale, which often helps
    optimisation and makes comparisons across case studies more stable.

- **Cell C - Summarise `value_scaled` by split (train / validation / test)**
  - Groups the data by `split` and reports `mean`, `std`, `min`, and `max` of `value_scaled` for each segment.
  - Shows how the scaled temperature distribution differs between training, validation, and test periods, which
    is useful for understanding **concept drift** and for describing the data in the methodology/results chapters.
  - Provides a table that documents the effect of scaling across all three splits.


## Reorder columns for a clean schema

In [43]:
column_order = [
    "time",
    "value",
    "value_scaled",
    "is_anomaly",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "split",
    "case_study",
]

# Reorder the dataframe using the defined schema
df = df[column_order]

# Quick check of the first few rows
df.head()

,time,value,value_scaled,is_anomaly,hour_of_day,day_of_week,is_weekend,split,case_study
0,2013-07-04 00:00:00,21.044908,-0.715797,0,0,3,0,train,ambient
1,2013-07-04 01:00:00,21.789015,-0.311577,0,1,3,0,train,ambient
2,2013-07-04 02:00:00,21.598781,-0.414918,0,2,3,0,train,ambient
3,2013-07-04 03:00:00,20.533000,-0.993880,0,3,3,0,train,ambient
4,2013-07-04 04:00:00,20.713084,-0.896054,0,4,3,0,train,ambient


## Recreate split-specific dataframes so they include `value_scaled`

In [44]:
train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()


In [45]:
#Quick check, rows should be equal
len(df), len(train_df) + len(validation_df) + len(test_df)

(7267, 7267)

## Step 10 – Save processed ambient datasets to disk

In [46]:
import os

# Folder for processed ambient data
output_dir = "../data/processed/ambient"
os.makedirs(output_dir, exist_ok=True)

# Save full processed dataframe (ground truth processed file)
full_path = os.path.join(output_dir, "ambient_processed_full.csv")
df.to_csv(full_path, index=False)

# Optional: save split-specific convenience files
train_path = os.path.join(output_dir, "ambient_train.csv")
val_path = os.path.join(output_dir, "ambient_validation.csv")
test_path = os.path.join(output_dir, "ambient_test.csv")

train_df.to_csv(train_path, index=False)
validation_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)


In [47]:
full_path, train_path, val_path, test_path


('../data/processed/ambient/ambient_processed_full.csv',
 '../data/processed/ambient/ambient_train.csv',
 '../data/processed/ambient/ambient_validation.csv',
 '../data/processed/ambient/ambient_test.csv')

## Save processed ambient datasets

The following CSV files are written to the `data/processed/ambient` folder:

- `ambient_processed_full.csv`  
  - Contains the **entire processed Ambient A dataset** after preprocessing.  
  - Includes all rows and all relevant columns:
    - `time`, `value` (°C), `value_scaled`, `is_anomaly`,
      `split`, `hour_of_day`, `day_of_week`, `is_weekend`, `case_study`.  
  - This is the **canonical processed file for Ambient A** and should be treated as the main reference
    dataset for analysis and modelling.

- `ambient_train.csv`  
  - Subset of `ambient_processed_full.csv` where `split == "train"`.  
  - Used for fitting models and computing preprocessing statistics.

- `ambient_validation.csv`  
  - Subset where `split == "validation"`.  
  - Used for model selection, hyperparameter tuning, and early comparison of methods.

- `ambient_test.csv`  
  - Subset where `split == "test"`.  
  - Held-out data used for **final evaluation** and reporting of results.

All three split-specific files are derived directly from `ambient_processed_full.csv`, ensuring a
consistent and reproducible link between the full processed dataset and the train/validation/test
segments used in experiments.
